# SONIC fine-tuning walkthrough

This notebook walks through the complete process of fine-tuning a SONIC policy for the Unitree G1 on the `s_batido` motion. First select the **SONIC (Python 3.11)** kernel in the upper-right corner of JupyterLab.

Long training runs use `tmux` so they continue if the notebook disconnects.

## 1. Check the GPU and training environment

In [ ]:
import subprocess
import sys
from pathlib import Path

import torch
import isaaclab

print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'], check=True)

## 2. Check paths and source motion

In [ ]:
PROJECT = Path('/workspace/ultimate-bots-G1')
SONIC = Path('/workspace/GR00T-WholeBodyControl')
SOURCE_MOTION = PROJECT / 'data/source/sonic/s_batido_test_sonic'
MOTION_LIB = PROJECT / 'data/motion_lib/s_batido_test.pkl'
CHECKPOINT = SONIC / 'sonic_release/last.pt'

required_files = [SOURCE_MOTION / name for name in ('joint_pos.csv', 'body_pos.csv', 'body_quat.csv')]
print('Studio SONIC motion files:', required_files)
assert all(path.exists() for path in required_files), 'The SONIC CSV bundle converted from Studio is required.'

## 3. Convert to a SONIC motion-lib PKL

Convert the G1 joint and body CSV bundle from Studio into the motion-lib PKL format used by SONIC. The Studio metadata specifies 50 FPS and 81 frames.

In [ ]:
convert_cmd = [
    sys.executable,
    str(SONIC / 'gear_sonic/data_process/convert_soma_csv_to_motion_lib.py'),
    '--input', str(SOURCE_MOTION),
    '--output', str(MOTION_LIB),
    '--fps', '50',
]
print(' '.join(convert_cmd))
subprocess.run(convert_cmd, cwd=SONIC, check=True)
print('Generated PKL:', MOTION_LIB, MOTION_LIB.stat().st_size, 'bytes')

## 4. Download the public SONIC checkpoint

In [ ]:
download_cmd = [sys.executable, 'download_from_hf.py', '--training', '--no-smpl']
subprocess.run(download_cmd, cwd=SONIC, check=True)
assert CHECKPOINT.exists(), f'Checkpoint not found: {CHECKPOINT}'
print('Base checkpoint:', CHECKPOINT, CHECKPOINT.stat().st_size / 1024**2, 'MiB')

## 5. Before: evaluate the base policy

Record metrics and video showing how the policy behaves on this motion before fine-tuning.

In [ ]:
before_dir = PROJECT / 'videos/before'
before_dir.mkdir(parents=True, exist_ok=True)
before_cmd = [
    sys.executable, 'gear_sonic/eval_agent_trl.py',
    f'+checkpoint={CHECKPOINT}',
    '+headless=True',
    '++eval_callbacks=im_eval',
    '++run_eval_loop=False',
    '++num_envs=1',
    '++manager_env.config.render_results=True',
    f'++manager_env.config.save_rendering_dir={before_dir}',
    '~manager_env/recorders=empty',
    '+manager_env/recorders=render',
    f'++manager_env.commands.motion.motion_lib_cfg.motion_file={MOTION_LIB}',
    '++manager_env.commands.motion.motion_lib_cfg.smpl_motion_file=dummy',
]
print(' '.join(map(str, before_cmd)))
# Uncomment after verifying the environment installation and data conversion.
# subprocess.run(before_cmd, cwd=SONIC, env={'ACCEPT_EULA': 'Y', **dict(__import__('os').environ)}, check=True)

## 6. Short fine-tuning smoke test

First run five iterations with 16 environments to verify that the full pipeline works.

In [ ]:
smoke_cmd = [
    sys.executable, 'gear_sonic/train_agent_trl.py',
    '+exp=manager/universal_token/all_modes/sonic_release',
    f'+checkpoint={CHECKPOINT}',
    'num_envs=16',
    'headless=True',
    '++algo.config.num_learning_iterations=5',
    f'++manager_env.commands.motion.motion_lib_cfg.motion_file={MOTION_LIB}',
    '++manager_env.commands.motion.motion_lib_cfg.smpl_motion_file=dummy',
    'use_wandb=false',
]
print(' '.join(map(str, smoke_cmd)))
# subprocess.run(smoke_cmd, cwd=SONIC, env={'ACCEPT_EULA': 'Y', **dict(__import__('os').environ)}, check=True)

## 7. Start the production fine-tuning run

Start training in a `tmux` session so it continues after the notebook closes. Begin with 1,024 environments on an A40, then adjust after checking GPU memory usage.

In [ ]:
train_command = ' '.join([
    sys.executable, 'gear_sonic/train_agent_trl.py',
    '+exp=manager/universal_token/all_modes/sonic_release',
    f'+checkpoint={CHECKPOINT}',
    'num_envs=1024', 'headless=True',
    '++algo.config.actor_learning_rate=5e-6',
    '++algo.config.desired_kl=0.005',
    f'++manager_env.commands.motion.motion_lib_cfg.motion_file={MOTION_LIB}',
    '++manager_env.commands.motion.motion_lib_cfg.smpl_motion_file=dummy',
    'wandb.wandb_project=ultimate-bots-g1',
])
log_file = PROJECT / 'train.log'
tmux_cmd = [
    'tmux', 'new-session', '-d', '-s', 'sonic_train',
    f"cd {SONIC} && ACCEPT_EULA=Y {train_command} 2>&1 | tee {log_file}",
]
print(' '.join(tmux_cmd))
# subprocess.run(tmux_cmd, check=True)

In [ ]:
# Inspect the latest training log.
if log_file.exists():
    print(''.join(log_file.read_text(errors='replace').splitlines(True)[-60:]))
else:
    print('train.log does not exist yet.')

## 8. After evaluation and ONNX export

After training, set `BEST_CHECKPOINT` to the checkpoint with the best evaluation score rather than automatically using the final file.

In [ ]:
BEST_CHECKPOINT = Path('/workspace/GR00T-WholeBodyControl/logs_rl/TRL_G1_Track/CHANGE_ME/model_step_002000.pt')
export_cmd = [
    sys.executable, 'gear_sonic/eval_agent_trl.py',
    f'+checkpoint={BEST_CHECKPOINT}',
    '+headless=True', '++num_envs=1', '+export_onnx_only=true',
]
print(' '.join(map(str, export_cmd)))
# subprocess.run(export_cmd, cwd=SONIC, env={'ACCEPT_EULA': 'Y', **dict(__import__('os').environ)}, check=True)